# Ejercicio 11: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
import os

In [2]:
ruta_archivo = "/kaggle/input/receta2/view-source_https___www.allrecipes.com_recipe_235997_unstuffed-cabbage-roll_.html"

In [3]:
# Usamos 'utf-8' para evitar problemas con tildes o caracteres especiales
with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
    contenido_html = archivo.read()

In [4]:
soup = BeautifulSoup(contenido_html, 'html.parser')

In [5]:
# Extracting the recipe title
title = soup.find("title")
title.string

'Unstuffed Cabbage Roll Recipe'

In [6]:
score=soup.find("div", {"id":"mm-recipes-review-bar__rating_1-0"})
score.string

'4.6'

## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [10]:
def parsearReceta(html_doc_path):
    with open(html_doc_path, 'r', encoding='utf-8') as archivo:
        html_doc = archivo.read()

    soup = BeautifulSoup(html_doc, 'html.parser')
    
    score=soup.find("div", {"id":"mm-recipes-review-bar__rating_1-0"}).string
    prep_time=soup.find("div", {"class":"mm-recipes-details__value"}).string
    description=soup.find("p", {"class":"article-subheading text-utility-300"}).string
    title = soup.find("title").string
    return {"score":score,"prep_time":prep_time, "description":description, "title":title}

In [7]:
description=soup.find("p", {"class":"article-subheading text-utility-300"})
description.string

"This is an easy casserole made with ground beef, cabbage, garlic, and tomatoes. My kids don't even like cabbage, but they love this dish! Serve with rice for a comforting weeknight dinner. Also, the longer it stands the better it tastes!"

In [8]:
prep_time=soup.find("div", {"class":"mm-recipes-details__value"})
prep_time.string

'15 mins'

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [13]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F235997%2Funstuffed-cabbage-roll%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://support.people.inc/hc/en-us/categories/360003648613-Allrecipes
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F235997%2Funstuffed-cabbage-roll%2F
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-stews-and-chili/
https://www.allrecipes.com/recipes/16099/everyday-cooking/comfort-food/
htt

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [24]:
!pip install langchain langchain-community langchain-core chromadb sentence-transformers

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [25]:
import requests
from bs4 import BeautifulSoup
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA
from langchain_community.llms import OpenAI

def parsearReceta_desde_web(html_content, url):
    soup = BeautifulSoup(html_content, 'html.parser')
    try:
        score_div = soup.find("div", {"id": lambda x: x and "rating" in x}) # Selector más flexible
        score = score_div.get_text(strip=True) if score_div else "N/A"
        
        prep_div = soup.find("div", {"class": "mm-recipes-details__value"})
        prep_time = prep_div.get_text(strip=True) if prep_div else "N/A"
        
        desc_p = soup.find("p", {"class": "article-subheading"})
        description = desc_p.get_text(strip=True) if desc_p else ""
        
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else "Sin título"
        
        # Formateamos el texto para que la IA lo entienda mejor
        texto_final = f"RECETA: {title}\nDESCRIPCIÓN: {description}\nTIEMPO: {prep_time}\nPUNTUACIÓN: {score}"
        
        return {"texto": texto_final, "titulo": title, "url": url}
    except Exception as e:
        print(f"Error parseando {url}: {e}")
        return None

In [27]:
import time
import requests
from langchain.docstore.document import Document

# 1. DEFINIMOS CABECERAS (Para parecer un humano navegando con Chrome)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

docs_para_rag = []

print(f"Analizando {len(recipe_urls)} enlaces...")

for url in recipe_urls:
    # 2. LIMPIEZA: Solo procesamos si es una URL completa y parece una receta
    # Las recetas de Allrecipes suelen tener "/recipe/" en la URL
    if not url.startswith("http"):
        continue # Saltamos errores de rutas relativas como '/account/add-recipe'
        
    if "facebook.com" in url or "instagram.com" in url or "google.com" in url:
        continue # Saltamos redes sociales
        
    # Opcional: Filtro estricto (Descomenta si solo quieres recetas puras)
    # if "/recipe/" not in url:
    #    continue 

    try:
        # 3. PETICIÓN CON MÁSCARA (Añadimos headers)
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            datos = parsearReceta_desde_web(response.text, url)
            
            # Verificamos que realmente capturamos un título válido
            if datos and datos["titulo"] not in ["N/A", "Sin título", "Log in to Facebook"]:
                doc = Document(
                    page_content=datos["texto"],
                    metadata={"source": datos["url"], "title": datos["titulo"]}
                )
                docs_para_rag.append(doc)
                print(f"✅ Éxito: {datos['titulo'][:50]}...") # Imprimimos solo el inicio para no ensuciar
            else:
                # A veces descarga bien (200) pero es una página de categoría sin receta
                pass 
                
        elif response.status_code == 460 or response.status_code == 403:
            print(f"⛔ Bloqueado (Anti-bot) en: {url}")
        else:
            print(f"⚠️ Status {response.status_code} en {url}")
            
        # 4. ÉTICA Y SEGURIDAD: Pausa pequeña para no saturar y que no nos vuelvan a bloquear
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error de conexión: {e}")

print(f"\nTotal documentos válidos listos para RAG: {len(docs_para_rag)}")

Analizando 203 enlaces...
✅ Éxito: Sign in to Allrecipes...
✅ Éxito: Sign in to MyRecipes...
⛔ Bloqueado (Anti-bot) en: https://support.people.inc/hc/en-us/categories/360003648613-Allrecipes
✅ Éxito: Unstuffed Cabbage Roll Recipe...
✅ Éxito: Dinner Recipes...
✅ Éxito: 5 Ingredient Main Dish Recipes...
✅ Éxito: One-Pot Meal Recipes...
✅ Éxito: Quick and Easy Recipes...
✅ Éxito: 30-Minute Meal Recipes...
✅ Éxito: Family Dinner Ideas & Recipes...
✅ Éxito: Soups, Stews and Chili Recipes...
✅ Éxito: Comfort Food Recipes...
✅ Éxito: Main Dishes...
✅ Éxito: Sheet Pan Dinner Recipes...
✅ Éxito: Dinner Recipes...
✅ Éxito: Recipes A-Z | Allrecipes.com...
✅ Éxito: Breakfast and Brunch Recipes...
✅ Éxito: Lunch Recipes...
✅ Éxito: Healthy Recipes...
✅ Éxito: Appetizers and Snacks...
✅ Éxito: Salad Recipes...
✅ Éxito: Side Dish Recipes...
✅ Éxito: Soup Recipes...
✅ Éxito: Bread Recipes...
✅ Éxito: Drinks Recipes...
✅ Éxito: Dessert Recipes...
✅ Éxito: Recipes A-Z | Allrecipes.com...
✅ Éxito: Ingred

In [29]:
import os
import warnings
import textwrap

# 1. LIMPIEZA DE RUIDO (Ejecutar esto ANTES de cargar el modelo si es posible)
# Silencia los logs de TensorFlow/CUDA (las letras rojas)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
# Ignora las advertencias de deprecación de LangChain
warnings.filterwarnings('ignore')

# (Tu configuración del modelo y retriever se mantiene igual aquí...)
# embedding_model = ...
# vector_db = ...
# retriever = ...

def buscar_receta_elegante(pregunta):
    # Traemos un poco más (k=4) para poder filtrar duplicados si aparecen
    docs_crudos = retriever.invoke(pregunta)
    
    print(f"\n🍳 BUSCANDO: '{pregunta}'")
    print("=" * 60)
    
    urls_vistas = set()
    contador = 1
    
    for doc in docs_crudos:
        # Extraemos metadatos
        titulo = doc.metadata.get('title', 'Sin título')
        link = doc.metadata.get('source', '#')
        
        # FILTRO ANTIDUPLICADOS: Si ya mostramos este link, pasamos al siguiente
        if link in urls_vistas:
            continue
        urls_vistas.add(link)
        
        # Limpiamos el contenido para que no se vea el "RECETA: ..." repetido
        # Asumimos que tu texto viene con etiquetas, así que mostramos una vista previa limpia
        contenido_preview = doc.page_content.replace("\n", " ")[:200] + "..."
        
        # IMPRESIÓN CON FORMATO MEJORADO
        print(f"Resultados #{contador}")
        print(f"📌 {titulo.upper()}")
        print(f"🔗 {link}")
        print(f"📖 Extracto: {textwrap.fill(contenido_preview, width=80)}")
        print("-" * 60)
        
        contador += 1
        if contador > 2: # Solo queremos mostrar los 2 mejores ÚNICOS
            break
            
    if contador == 1:
        print("❌ No se encontraron recetas relevantes para tu búsqueda.")

# --- PRUEBA DEL SISTEMA ---
buscar_receta_elegante("¿Hay alguna receta que sea rápida de hacer?")
buscar_receta_elegante("Quiero cocinar algo dulce o postre")


🍳 BUSCANDO: '¿Hay alguna receta que sea rápida de hacer?'
Resultados #1
📌 FAST FOOD
🔗 https://www.allrecipes.com/fast-food-8672832
📖 Extracto: RECETA: Fast Food DESCRIPCIÓN:  TIEMPO: N/A PUNTUACIÓN: N/A...
------------------------------------------------------------

🍳 BUSCANDO: 'Quiero cocinar algo dulce o postre'
Resultados #1
📌 BREAD RECIPES
🔗 https://www.allrecipes.com/recipes/156/bread/
📖 Extracto: RECETA: Bread Recipes DESCRIPCIÓN:  TIEMPO: N/A PUNTUACIÓN: ...
------------------------------------------------------------
